# Models

> Core data models for the file browser including DirectoryListing, BrowserSelection, and BrowserState.

In [ ]:
#| default_exp core.models

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from cjm_file_discovery.core.models import FileInfo

## DirectoryListing

Result of listing a directory, including the items and metadata about the listing operation.

In [ ]:
#| export
@dataclass
class DirectoryListing:
    """Result of listing a directory."""
    path: str                              # Current directory path
    items: List[FileInfo]                  # Directory contents
    parent_path: Optional[str] = None      # Parent directory (None if at root)
    provider_name: str = "local"           # Source provider
    total_items: int = 0                   # Total count (for pagination)
    error: Optional[str] = None            # Error message if listing failed
    
    def __post_init__(self):
        """Set total_items from items length if not provided."""
        if self.total_items == 0 and self.items:
            self.total_items = len(self.items)

In [ ]:
from cjm_file_discovery.core.models import FileType

# Test DirectoryListing
listing = DirectoryListing(
    path="/home/user",
    items=[
        FileInfo(name="file.txt", path="/home/user/file.txt", is_directory=False),
        FileInfo(name="folder", path="/home/user/folder", is_directory=True),
    ],
    parent_path="/home"
)

assert listing.path == "/home/user"
assert len(listing.items) == 2
assert listing.total_items == 2
assert listing.parent_path == "/home"
assert listing.error is None

# Test with error
error_listing = DirectoryListing(
    path="/restricted",
    items=[],
    error="Permission denied"
)
assert error_listing.error == "Permission denied"

print("DirectoryListing tests passed!")

DirectoryListing tests passed!


## BrowserSelection

Tracks the current selection state in the browser, supporting single and multiple selection modes.

In [ ]:
#| export
@dataclass
class BrowserSelection:
    """Current selection state."""
    selected_paths: List[str] = field(default_factory=list)  # Selected file paths
    last_selected: Optional[str] = None    # Most recently selected (for shift-click)

    def add(
        self,
        path: str  # Path to add to selection
    ) -> None:  # Modifies selection in place
        """Add a path to the selection."""
        if path not in self.selected_paths:
            self.selected_paths.append(path)
        self.last_selected = path

    def remove(
        self,
        path: str  # Path to remove from selection
    ) -> None:  # Modifies selection in place
        """Remove a path from the selection."""
        if path in self.selected_paths:
            self.selected_paths.remove(path)
        if self.last_selected == path:
            self.last_selected = self.selected_paths[-1] if self.selected_paths else None

    def clear(self) -> None:  # Modifies selection in place
        """Clear all selections."""
        self.selected_paths.clear()
        self.last_selected = None

    def toggle(
        self,
        path: str  # Path to toggle
    ) -> None:  # Modifies selection in place
        """Toggle selection of a path."""
        if path in self.selected_paths:
            self.remove(path)
        else:
            self.add(path)

    def is_selected(
        self,
        path: str  # Path to check
    ) -> bool:  # True if path is selected
        """Check if a path is selected."""
        return path in self.selected_paths
    
    def set_single(
        self,
        path: str  # Path to set as single selection
    ) -> None:  # Modifies selection in place
        """Set a single path as the only selection (for single-select mode)."""
        self.selected_paths = [path]
        self.last_selected = path

In [ ]:
# Test BrowserSelection
selection = BrowserSelection()

# Test add
selection.add("/path/to/file1.txt")
assert selection.is_selected("/path/to/file1.txt")
assert selection.last_selected == "/path/to/file1.txt"

selection.add("/path/to/file2.txt")
assert len(selection.selected_paths) == 2
assert selection.last_selected == "/path/to/file2.txt"

# Test duplicate add (should not add twice)
selection.add("/path/to/file1.txt")
assert len(selection.selected_paths) == 2

# Test remove
selection.remove("/path/to/file1.txt")
assert not selection.is_selected("/path/to/file1.txt")
assert len(selection.selected_paths) == 1

# Test toggle
selection.toggle("/path/to/file2.txt")  # Remove
assert not selection.is_selected("/path/to/file2.txt")
selection.toggle("/path/to/file2.txt")  # Add back
assert selection.is_selected("/path/to/file2.txt")

# Test clear
selection.add("/path/to/file3.txt")
selection.clear()
assert len(selection.selected_paths) == 0
assert selection.last_selected is None

# Test set_single
selection.add("/path/to/file1.txt")
selection.add("/path/to/file2.txt")
selection.set_single("/path/to/only.txt")
assert len(selection.selected_paths) == 1
assert selection.is_selected("/path/to/only.txt")

print("BrowserSelection tests passed!")

BrowserSelection tests passed!


## BrowserState

Complete browser state for persistence and restoration. This captures everything needed to restore the browser to a specific state.

In [ ]:
#| export
@dataclass
class BrowserState:
    """Complete browser state for persistence/restore."""
    current_path: str                                          # Current directory path
    selection: BrowserSelection = field(default_factory=BrowserSelection)  # Selection state
    sort_by: str = "name"                                      # Sort field
    sort_descending: bool = False                              # Sort direction
    filter_extensions: Optional[List[str]] = None              # Active extension filter
    
    def to_dict(self) -> Dict[str, Any]:  # Serializable dictionary
        """Convert state to a serializable dictionary."""
        return {
            "current_path": self.current_path,
            "selected_paths": self.selection.selected_paths,
            "last_selected": self.selection.last_selected,
            "sort_by": self.sort_by,
            "sort_descending": self.sort_descending,
            "filter_extensions": self.filter_extensions,
        }
    
    @classmethod
    def from_dict(
        cls,
        data: Dict[str, Any]  # Dictionary with state data
    ) -> "BrowserState":  # Restored BrowserState instance
        """Create a BrowserState from a dictionary."""
        selection = BrowserSelection(
            selected_paths=data.get("selected_paths", []),
            last_selected=data.get("last_selected")
        )
        return cls(
            current_path=data["current_path"],
            selection=selection,
            sort_by=data.get("sort_by", "name"),
            sort_descending=data.get("sort_descending", False),
            filter_extensions=data.get("filter_extensions"),
        )

In [ ]:
# Test BrowserState
state = BrowserState(
    current_path="/home/user",
    sort_by="modified",
    sort_descending=True,
    filter_extensions=[".py", ".txt"]
)

state.selection.add("/home/user/file.txt")

assert state.current_path == "/home/user"
assert state.sort_by == "modified"
assert state.sort_descending == True
assert state.filter_extensions == [".py", ".txt"]
assert state.selection.is_selected("/home/user/file.txt")

# Test serialization round-trip
state_dict = state.to_dict()
restored_state = BrowserState.from_dict(state_dict)

assert restored_state.current_path == state.current_path
assert restored_state.sort_by == state.sort_by
assert restored_state.sort_descending == state.sort_descending
assert restored_state.filter_extensions == state.filter_extensions
assert restored_state.selection.selected_paths == state.selection.selected_paths

# Test defaults
default_state = BrowserState(current_path="/")
assert default_state.sort_by == "name"
assert default_state.sort_descending == False
assert default_state.filter_extensions is None

print("BrowserState tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()